# Optimizer Fine-tuning

In this section, we will fine-tune the optimizer for better performance. Fine-tuning the optimizer can help improve convergence and achieve better results in training your model.

In [3]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import plotly.graph_objects as go

sys.path.append(os.path.abspath(".."))

from src.airfoil_predictor import AirfoilOptimizer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
optimizer = AirfoilOptimizer(chord=1.0)

best, finesse = optimizer.find_best_airfoil(185.5, 4200, n_trials=1000, show_history=True)

print(f"✅ Best NACA Profile : {best['m']}{best['p']}{int(best['t']):02d} with L/D = {finesse:.2f}")

📊 Study completed in 1000 trials.


✅ Best NACA Profile : 9605 with L/D = 421.41


## Early Stopping

Early stopping is a technique used to prevent overfitting by stopping the training process when the model's performance on a validation set starts to degrade. This can be implemented by monitoring the validation loss and stopping training when it stops improving.

In [5]:
optimizer = AirfoilOptimizer(chord=1.0)

best, finesse = optimizer.find_best_airfoil(185.5, 4200, n_trials=1000, show_history=True, early_stop=200)

print(f"✅ Best NACA Profile : {best['m']}{best['p']}{int(best['t']):02d} with L/D = {finesse:.2f}")

📊 Study completed in 288 trials.


✅ Best NACA Profile : 9605 with L/D = 421.41


In [6]:
def run_multi_early_stop_study(optimizer, v, alt, steps=[20, 40, 60, 80, 100], early_stops=[None, 10, 20], n_reps=20):
    """
    Compare the reliability of the optimizer for different early_stopping settings.
    """
    print(f"--- Calculating the absolute reference ---")
    _, l_d_ref = optimizer.find_best_airfoil(v, alt, n_trials=400, early_stop=None, show_history=False)
    target_value = l_d_ref * 0.99
    
    fig = go.Figure()
    colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']

    for i, es in enumerate(early_stops):
        label = f"No Early Stop" if es is None else f"Early Stop = {es}"
        reliability_results = []
        
        print(f"\nAnalysis: {label}")
        
        for n in steps:
            success_count = 0
            for _ in range(n_reps):
                _, current_finesse = optimizer.find_best_airfoil(v, alt, n_trials=n, early_stop=es, show_history=False)
                if current_finesse >= target_value:
                    success_count += 1
            
            reliability = (success_count / n_reps) * 100
            reliability_results.append(reliability)
            print(f"  Max Trials {n}: {reliability}% success")

        fig.add_trace(go.Scatter(
            x=steps, 
            y=reliability_results,
            mode='lines+markers',
            name=label,
            line=dict(width=3, color=colors[i % len(colors)]),
            marker=dict(size=8)
        ))

    fig.add_shape(type="line", x0=min(steps), x1=max(steps), y0=95, y1=95,
                  line=dict(color="black", width=2, dash="dot"), name="95% Threshold")

    fig.update_layout(
        title="Reliability Comparison: Influence of Early Stopping",
        xaxis_title="Computational Budget (max n_trials)",
        yaxis_title="Success Rate (%)",
        legend_title="Configuration",
        template="plotly_white",
        height=600
    )

    return fig

In [7]:

fig_multi = run_multi_early_stop_study(optimizer, 180.0, 4500.0, steps=[80, 100, 120, 140, 160, 180], early_stops=[None, 100, 50], n_reps=30)
fig_multi.show()

--- Calculating the absolute reference ---

Analysis: No Early Stop
  Max Trials 80: 56.666666666666664% success
  Max Trials 100: 90.0% success
  Max Trials 120: 90.0% success
  Max Trials 140: 96.66666666666667% success
  Max Trials 160: 96.66666666666667% success
  Max Trials 180: 100.0% success

Analysis: Early Stop = 100
  Max Trials 80: 53.333333333333336% success
  Max Trials 100: 90.0% success
  Max Trials 120: 86.66666666666667% success
  Max Trials 140: 100.0% success
  Max Trials 160: 96.66666666666667% success
  Max Trials 180: 93.33333333333333% success

Analysis: Early Stop = 50
  Max Trials 80: 66.66666666666666% success
  Max Trials 100: 60.0% success
  Max Trials 120: 76.66666666666667% success
  Max Trials 140: 63.33333333333333% success
  Max Trials 160: 83.33333333333334% success
  Max Trials 180: 96.66666666666667% success
